# Intro to `gwrefpy`

`gwrefpy` is a Python package for **detecting deviations in groundwater time series**.

This notebook can be downloaded from the source code [here](https://github.com/sgfsweden/gwrefpy/blob/main/docs/user_guide/1_intro_gwrefpy.ipynb).

```{admonition} Problem Statement
:class: note
**The problem:** Groundwater levels rise and fall naturally with weather and the seasons. So when an observation well behaves unexpectedly, how do we tell whether human activity (pumping, construction, drainage) is to blame, or whether it is just natural variation?

**How `gwrefpy` solves it:** An **observation well** (the one we care about) is compared against one or more **reference wells** that represent undisturbed, natural conditions. `gwrefpy` fits the observation well to the reference wells by regression; once later measurements stop following that fit, a deviation — some kind of influence — has occurred.
```

The two building blocks of `gwrefpy` are **`Model`** — a site or area of interest — and **`Well`** — a location where groundwater data is collected. A `Model` holds any number of `Well` objects, and the two can be created independently and linked together later.

Let's start by importing `gwrefpy` (along with `pandas`, which we use to load the data).

In [1]:
import pandas as pd

import gwrefpy as gr

# Creating a Model

A **`Model`** represents a site or area of interest and is made up of any number of wells. Here we create one named `"My First gwrefpy Model"`.

In [2]:
model = gr.Model(name="My First gwrefpy Model")

# Creating Wells

A **`Well`** is a location where groundwater data is collected. Each well is either an *observation* well or a *reference* well.

```{glossary}
*Reference wells*
    Wells that are used to represent natural conditions.

*Observation wells*
    Wells that the user is interested in to see if they are influenced by anthropogenic activities.
```

We will create two observation wells and two reference wells and later add them to our model.

## Creating data timeseries

A well's data is a `pandas` time series. In this example we build those series from CSV files.

```{tip}
Data types can be other than CSV. Check the pandas documentation for inspiration.
```

In [3]:
base_url = "https://github.com/sgfsweden/gwrefpy/raw/dev/docs/user_guide/data"


def load_series(filename):
    df = pd.read_csv(
        f"{base_url}/{filename}",
        sep=";",
        index_col=0,
        parse_dates=True,
    )
    # Each CSV has a single value column; take it as a pandas Series.
    return df.iloc[:, 0]


obs1_series = load_series("obs1_mod_wel.csv")
ref1_series = load_series("ref1_mod_wel.csv")
obs2_series = load_series("obs2_mod_wel.csv")
ref2_series = load_series("ref2_mod_wel.csv")

Next we create the `Well` objects and attach the time series to them.

```{admonition} The Well object
:class: note

`Well` takes three arguments: a `name` (string), `is_reference` (a boolean choosing reference or observation well), and the `timeseries` (the pandas series we created earlier).
```

In [4]:
obs1 = gr.Well(name="obs1", is_reference=False, timeseries=obs1_series)
ref1 = gr.Well(name="ref1", is_reference=True, timeseries=ref1_series)
obs2 = gr.Well(name="obs2", is_reference=False, timeseries=obs2_series)
ref2 = gr.Well(name="ref2", is_reference=True, timeseries=ref2_series)

Now we add the wells to the model.

# Add wells to the model

We add wells with `add_well()`, passing either a single well or a list of wells.

In [5]:
# Add several wells at once with a list
model.add_well([obs1, ref1, obs2])

# Add a single well
model.add_well(ref2)

Now let's confirm the wells are stored in the model.

# Check which wells are included in the model

The `obs_wells` and `ref_wells` properties list the observation and reference wells in the model.

In [6]:
model.obs_wells

[Well(name=obs1), Well(name=obs2)]

Check the reference wells:

In [7]:
model.ref_wells

[Well(name=ref1), Well(name=ref2)]

We can look up specific wells by name with `get_wells()`, passing one name or a list of names.

In [8]:
model.get_wells("obs1")
model.get_wells(["obs1", "ref1"])

[Well(name=obs1), Well(name=ref1)]

The `well_names` property returns just the names of the wells in the model.

In [9]:
model.well_names

['obs1', 'ref1', 'obs2', 'ref2']

The model now holds two observation wells and two reference wells, ready for further analysis.

For an overview, `wells_summary()` returns a `pandas` DataFrame summarising every well.

In [10]:
model.wells_summary()

,name,well_type,data_points,start_date,end_date,mean_level,latest_value,latest_date,latitude,longitude,elevation,best_fit_ref_well,best_rmse,num_fits,avg_rmse
0,obs1,observation,2078,2020-01-01,2025-09-08,15.467492,14.877000,2025-09-08,None,None,None,None,None,NaN,NaN
1,obs2,observation,2078,2020-01-01,2025-09-08,14.977455,14.832391,2025-09-08,None,None,None,None,None,NaN,NaN
2,ref1,reference,2078,2020-01-01,2025-09-08,25.799352,24.795000,2025-09-08,None,None,None,NaN,NaN,0.0,None
3,ref2,reference,2079,2020-01-01,2025-09-08,25.666046,25.900000,NaT,None,None,None,NaN,NaN,0.0,None


# Delete wells from the model

We remove wells with `delete_well()`, passing a `Well` object (or a list of them).

In [11]:
model.delete_well([ref1])

# Confirm it was removed
model.well_names

['obs1', 'obs2', 'ref2']

That's it for the intro to `gwrefpy`.

The next notebooks build on this model to cover the rest of `gwrefpy`: pairing observation and reference wells, fitting them, and plotting the results.